In [4]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from tqdm import tqdm

# 开启进度条
tqdm.pandas()

def clean_oe62_with_logp(json_file="df_62k.json", output_csv="opv_rl_4props.csv"):
    print(f"🚀 正在读取 {json_file} ...")
    try:
        df = pd.read_json(json_file, orient='split')
    except ValueError:
        try:
            df = pd.read_json(json_file)
        except:
            print("❌ 读取失败，请检查文件。")
            return

    print(f"✅ 原始数据量: {len(df)}")
    print("   正在提取 4 大核心性质 (HOMO, Gap, MolMR, LogP)...")

    # ==========================================
    # 1. 提取 DFT 能级
    # ==========================================
    col_occ = 'energies_occ_pbe0_vac_tier2'
    col_unocc = 'energies_unocc_pbe0_vac_tier2'
    col_smiles = 'canonical_smiles'

    if col_occ not in df.columns:
        print("⚠️ 没找到 energies_occ_pbe0_vac_tier2，尝试模糊搜索...")
        # 简单的自动回退机制
        for c in df.columns:
            if 'energies_occ' in c: col_occ = c
            if 'energies_unocc' in c: col_unocc = c
            if 'smiles' in c: col_smiles = c
            
    # 提取函数
    def get_homo(val): return val[-1] if isinstance(val, list) and len(val)>0 else None
    def get_lumo(val): return val[0] if isinstance(val, list) and len(val)>0 else None

    # ==========================================
    # 2. RDKit 计算函数 (MolMR + LogP)
    # ==========================================
    def calc_rdkit_props(smiles):
        try:
            mol = Chem.MolFromSmiles(smiles)
            if not mol: return None, None
            
            # A. MolMR (极化率替身) -> Maximize
            mr = Descriptors.MolMR(mol)
            
            # B. LogP (脂溶性/加工性) -> Maximize (OPV需要疏水)
            logp = Descriptors.MolLogP(mol)
            
            return mr, logp
        except:
            return None, None

    # ==========================================
    # 3. 执行处理
    # ==========================================
    new_df = pd.DataFrame()
    new_df['smiles'] = df[col_smiles]

    # DFT 部分
    print("   -> 提取 HOMO / Gap...")
    homo = df[col_occ].apply(get_homo)
    lumo = df[col_unocc].apply(get_lumo)
    new_df['homo'] = homo
    new_df['gap'] = lumo - homo

    # RDKit 部分
    print("   -> 计算 MolMR 和 LogP...")
    props = new_df['smiles'].progress_apply(calc_rdkit_props)
    
    new_df['MolMR'] = props.apply(lambda x: x[0] if x else None)
    new_df['LogP']  = props.apply(lambda x: x[1] if x else None)

    # ==========================================
    # 4. 保存
    # ==========================================
    df_clean = new_df.dropna()
    df_clean.to_csv(output_csv, index=False)
    
    print(f"\n✅ 处理完成！")
    print(f"   文件: {output_csv}")
    print(f"   数据量: {len(df_clean)}")
    print(f"   包含列: {list(df_clean.columns)}")
    print("-" * 40)
    print("【RL 奖励函数配置建议】")
    print("1. homo  -> Minimize ⬇️ (电压 Voc)")
    print("2. gap   -> Minimize ⬇️ (电流 Jsc)")
    print("3. MolMR -> Maximize ⬆️ (电荷传输)")
    print("4. LogP  -> Maximize ⬆️ (加工溶解性)")

if __name__ == "__main__":
    clean_oe62_with_logp()

🚀 正在读取 df_62k.json ...
✅ 原始数据量: 61489
   正在提取 4 大核心性质 (HOMO, Gap, MolMR, LogP)...
   -> 提取 HOMO / Gap...
   -> 计算 MolMR 和 LogP...


  0%|          | 0/61489 [00:00<?, ?it/s][20:31:35] Can't kekulize mol.  Unkekulized atoms: 2 3 4 5 17
[20:31:35] Explicit valence for atom # 3 N, 4, is greater than permitted
[20:31:35] Explicit valence for atom # 2 C, 5, is greater than permitted
[20:31:35] Explicit valence for atom # 3 N, 4, is greater than permitted
[20:31:35] Explicit valence for atom # 4 N, 4, is greater than permitted
  1%|▏         | 895/61489 [00:00<00:20, 2952.41it/s][20:31:35] Can't kekulize mol.  Unkekulized atoms: 9 10 11
[20:31:35] Explicit valence for atom # 2 C, 5, is greater than permitted
[20:31:35] Explicit valence for atom # 16 N, 4, is greater than permitted
  2%|▏         | 1197/61489 [00:00<00:20, 2978.82it/s][20:31:35] Explicit valence for atom # 5 N, 4, is greater than permitted
[20:31:35] Can't kekulize mol.  Unkekulized atoms: 13 14 16 17 18
[20:31:35] Explicit valence for atom # 2 N, 4, is greater than permitted
[20:31:35] Can't kekulize mol.  Unkekulized atoms: 12 13 14
  2%|▏         | 150


✅ 处理完成！
   文件: opv_rl_4props.csv
   数据量: 61037
   包含列: ['smiles', 'homo', 'gap', 'MolMR', 'LogP']
----------------------------------------
【RL 奖励函数配置建议】
1. homo  -> Minimize ⬇️ (电压 Voc)
2. gap   -> Minimize ⬇️ (电流 Jsc)
3. MolMR -> Maximize ⬆️ (电荷传输)
4. LogP  -> Maximize ⬆️ (加工溶解性)
